In [11]:
import tensorflow as tf
import numpy as np
from datasets import load_dataset
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

In [12]:

# =============================
# PARAMETERS
# =============================
IMG_SIZE = 224
BATCH_SIZE = 8
EPOCHS = 5
NUM_CLASSES = 101

In [13]:
# =============================
# 1️⃣ Load Dataset
# =============================
dataset = load_dataset("food101")

train_data = dataset["train"]
val_data = dataset["validation"]

In [14]:
# =============================
# 2️⃣ Preprocess Function
# =============================
def preprocess(example):
    image = example["image"].convert("RGB")  # Fix grayscale issue
    image = image.resize((IMG_SIZE, IMG_SIZE))
    image = np.array(image, dtype=np.float32) / 255.0
    label = example["label"]
    return image, label

In [15]:
# =============================
# 3️⃣ Generator
# =============================
def generator(dataset_split):
    for example in dataset_split:
        yield preprocess(example)

In [16]:
# =============================
# 4️⃣ Convert to TF Dataset
# =============================
output_signature = (
    tf.TensorSpec(shape=(IMG_SIZE, IMG_SIZE, 3), dtype=tf.float32),
    tf.TensorSpec(shape=(), dtype=tf.int64),
)

train_tf = tf.data.Dataset.from_generator(
    lambda: generator(train_data),
    output_signature=output_signature
)

val_tf = tf.data.Dataset.from_generator(
    lambda: generator(val_data),
    output_signature=output_signature
)

train_tf = train_tf.shuffle(2000).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_tf = val_tf.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

In [17]:
# =============================
# 5️⃣ Load Pretrained MobileNetV2
# =============================
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_SIZE, IMG_SIZE, 3)
)

base_model.trainable = False  # Freeze initially

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=output)

In [18]:
# =============================
# 6️⃣ Compile
# =============================
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 224, 224,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1 (Conv2D)      │ (None, 112, 112,  │        864 │ input_layer_1[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_Conv1            │ (None, 112, 112,  │        128 │ Conv1[0][0]       │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Conv1_relu (ReLU)   │ (None, 112, 112,  │          0 │ bn_Conv1[0][0]    │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        288 │ Conv1_relu[0][0]  │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │        128 │ expanded_conv_de… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_dept… │ (None, 112, 112,  │          0 │ expanded_conv_de… │
│ (ReLU)              │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │        512 │ expanded_conv_de… │
│ (Conv2D)            │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expanded_conv_proj… │ (None, 112, 112,  │         64 │ expanded_conv_pr… │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand      │ (None, 112, 112,  │      1,536 │ expanded_conv_pr… │
│ (Conv2D)            │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_BN   │ (None, 112, 112,  │        384 │ block_1_expand[0… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_expand_relu │ (None, 112, 112,  │          0 │ block_1_expand_B… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_pad         │ (None, 113, 113,  │          0 │ block_1_expand_r… │
│ (ZeroPadding2D)     │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise   │ (None, 56, 56,    │        864 │ block_1_pad[0][0] │
│ (DepthwiseConv2D)   │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │        384 │ block_1_depthwis… │
│ (BatchNormalizatio… │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_depthwise_… │ (None, 56, 56,    │          0 │ block_1_depthwis… │
│ (ReLU)              │ 96)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block_1_project     │ (None, 56, 56,    │      2,304 │ block_1_depthwis

 Total params: 2,611,877 (9.96 MB)

 Trainable params: 353,893 (1.35 MB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [19]:

# =============================
# 7️⃣ Callbacks
# =============================
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=2,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "best_food101_model.h5",
    monitor="val_accuracy",
    save_best_only=True
)

In [20]:
# =============================
# 8️⃣ Train
# =============================
model.fit(
    train_tf,
    validation_data=val_tf,
    epochs=EPOCHS,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/5
   3514/Unknown 303s 83ms/step - accuracy: 0.6761 - loss: 1.2484

c:\Users\Chandubhai Chotaliya\Desktop\AI\Live Project\Deep Learning\foodenv\Lib\site-packages\PIL\TiffImagePlugin.py:949: UserWarning: Truncated File Read
  warnings.warn(str(msg))


   9469/Unknown 786s 82ms/step - accuracy: 0.5191 - loss: 2.0790

c:\Users\Chandubhai Chotaliya\Desktop\AI\Live Project\Deep Learning\foodenv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:164: UserWarning: Your input ran out of data; interrupting training. Make sure that your dataset or generator can generate at least `steps_per_epoch * epochs` batches. You may need to use the `.repeat()` function when building your dataset.
  self._interrupted_warning()


9469/9469 ━━━━━━━━━━━━━━━━━━━━ 1031s 108ms/step - accuracy: 0.3792 - loss: 2.8652 - val_accuracy: 0.0631 - val_loss: 10.4231
Epoch 2/5
9469/9469 ━━━━━━━━━━━━━━━━━━━━ 1004s 105ms/step - accuracy: 0.4329 - loss: 2.2151 - val_accuracy: 0.0621 - val_loss: 17.1046
Epoch 3/5
9469/9469 ━━━━━━━━━━━━━━━━━━━━ 1711s 180ms/step - accuracy: 0.4491 - loss: 2.1594 - val_accuracy: 0.0605 - val_loss: 19.0496


In [21]:
# =============================
# 9️⃣ Optional Fine-Tuning
# =============================
print("🔓 Starting Fine-Tuning...")

base_model.trainable = True

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    train_tf,
    validation_data=val_tf,
    epochs=3
)


🔓 Starting Fine-Tuning...
Epoch 1/3
9469/9469 ━━━━━━━━━━━━━━━━━━━━ 5293s 556ms/step - accuracy: 0.1479 - loss: 3.2926 - val_accuracy: 0.0985 - val_loss: 4.6631
Epoch 2/3
9469/9469 ━━━━━━━━━━━━━━━━━━━━ 4173s 440ms/step - accuracy: 0.2316 - loss: 2.8274 - val_accuracy: 0.1193 - val_loss: 4.5891
Epoch 3/3
9469/9469 ━━━━━━━━━━━━━━━━━━━━ 5251s 554ms/step - accuracy: 0.3159 - loss: 2.4299 - val_accuracy: 0.1470 - val_loss: 4.3820


In [22]:
# =============================
# 🔟 Save Final Model
# =============================
model.save("final_trained_model.h5")

print("✅ Model Training Completed & Saved Successfully!")

✅ Model Training Completed & Saved Successfully!
